# jupyter-tts-alerts: verification notebook

Run this once with your volume up and you will have checked every part of the package.

Each section says what you should **hear**, so you can tell a working install from a silent
one without reading any code. Nothing here takes longer than a few seconds.

**Before you start**

1. Install a speech engine if you have not already: `sudo apt install espeak-ng` on Linux,
   nothing at all on macOS or Windows.
2. Install the package: `pip install jupyter-tts-alerts`, or `pip install .` from a clone of
   the repository.
3. **Run the cells one at a time** (Shift+Enter), and wait for each announcement before the
   next. Speech is queued and never talks over itself, and the queue keeps only the newest few
   lines, so racing through with Run All would drop some of the sentences you are listening for.

## 0. Import it

That is the entire setup. Importing the package hooks it into the kernel; there is no
`enable()` call to remember and nothing to add to your own notebooks.

**You should hear "Starting notebook."**, the confirmation that your speakers, the engine and
the kernel are all on the same machine. If you see the banner but hear nothing, section 1 will
tell you why.

In [ ]:
# %pip install jupyter-tts-alerts
import jupyter_tts_alerts as tts

## 1. What does it think it is doing?

`status()` is the first thing to check when nothing is coming out of the speakers. It names the
engine it found, the two voices it will use, and warns you if the kernel looks like it is running
on a machine you are not sitting at: a cluster node will happily talk to an empty room.

A line reading `engine  none found` means no speech engine is installed; everything below will
run silently but correctly.

In [ ]:
tts.status()

## 2. Can you hear both voices?

**You should hear two more sentences, in two noticeably different voices.** The second one is
the voice reserved for failures: the point is that you can tell a crash from a completion through a
closed door, without listening to the words.

In [ ]:
tts.test()

## 3. Fast cells stay quiet

**You should hear nothing.** Anything under the threshold (five seconds by default) passes
without comment, so writing code is not narrated back at you.

In [ ]:
sum(range(2_000_000))

## 4. Slow cells announce themselves

Dropping the threshold to two seconds so this notebook does not take all afternoon.

**You should hear "Cell *n*. 2 seconds."**, and *n* is the `In [n]` number beside the cell,
so you can find the thing that finished.

In [ ]:
tts.configure(min_seconds=2)

import time
time.sleep(2.5)

## 5. Silencing one cell

A `# tts: off` comment anywhere in a cell keeps it quiet, however long it takes. Useful for the
cell you re-run twenty times while you get it right.

**You should hear nothing**, despite this taking longer than the threshold.

In [ ]:
# tts: off
import time
time.sleep(2.5)

## 6. Forcing an announcement

The opposite: `# tts: always` announces a cell however fast it was.

**You should hear "Cell *n*. less than a second."**

In [ ]:
# tts: always
1 + 1

## 7. Saying something specific

Anything else after `# tts:` is used as the sentence. Worth doing on the cell that matters: a
sentence you recognise beats a cell number when you are in the next room.

**You should hear "the model has finished training".**

In [ ]:
# tts: the model has finished training
sum(range(100_000))

## 8. The `%tts` magic

Everything the Python API does, without the import line. `%tts help` lists the lot.

**You should hear "this one came from a magic".** The other two cells only print the
current settings.

In [ ]:
%tts

In [ ]:
%tts say this one came from a magic

In [ ]:
%tts set min_seconds=2 volume=60

## 9. Your own phrasing

Every sentence is a template. The fields are `{cell}`, `{duration}`, `{seconds}`, `{total}`,
`{total_seconds}`, `{error_type}` and `{error}`.

**You should hear "Cell *n* finally finished, after 2 seconds."**

In [ ]:
tts.configure(phrases={"cell": "Cell {cell} finally finished, after {duration}."})

import time
time.sleep(2.5)

In [ ]:
# Put the stock phrasing back.
tts.configure(phrases={"cell": "Cell {cell}. {duration}."});

## 10. Desktop notifications

Speech tells you a cell finished while you are making coffee; a notification tells you *which*
one once you are back at the screen. Off by default.

**You should hear the usual announcement and see a notification appear.**

In [ ]:
tts.configure(notify=True)

import time
time.sleep(2.5)

In [ ]:
tts.configure(notify=False);

## 11. Speaking from inside a long cell

`say()` talks immediately, whatever the thresholds are set to. Drop it in a training loop and
you get progress reports from the kitchen.

**You should hear three short progress announcements.**

In [ ]:
import time

for epoch in range(1, 4):
    time.sleep(0.4)
    tts.say(f"epoch {epoch} of 3 finished")

## 12. The finale

`# tts: final` on your last cell announces the total runtime of the whole session: the number
you actually want when you come back to a notebook that has been running since breakfast.

**You should hear the cell announcement, then "Notebook complete. Total runtime ...".**

In [ ]:
# tts: final
import time
time.sleep(2.5)

---

## Appendix: failures

The next two cells raise on purpose. Wait for each announcement before running the next cell:
the last cell switches the package off, which also drops anything still waiting to be said.

### A failing cell

**You should hear "Error in cell *n*. Value error." in the second voice**, however fast
the cell was. Errors skip the threshold, because a crash forty minutes into a six-hour run is the
one thing you want to know about straight away.

In [ ]:
raise ValueError("this failure is deliberate")

### An interrupt is not a failure

If you stopped the cell yourself, you are sitting right there and do not need telling.

**You should hear nothing.** (`tts.configure(announce_interrupts=True)` if you disagree.)

In [ ]:
raise KeyboardInterrupt()

## Turning it off

`tts.disable()` unhooks it completely; `%tts on` puts it back. To keep it quiet everywhere
without editing a notebook, set `JUPYTER_TTS_ENABLED=0` in the environment, or
`JUPYTER_TTS_AUTOSTART=0` to stop importing it from hooking anything up at all.

In [ ]:
tts.disable()
tts.status()